In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd

In [ ]:
test_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(test_path)
df

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here: 5. Plot the target distribution (delivery_time)
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery_Time distribution')
plt.xlabel('Delivery')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop(['Order_ID'], axis=1)
df

In [ ]:
# Task 2: Write your code here:
df.isnull().sum()
# df.describe() ##I choose filling with mean and unknown since the missing values doesnt seem alot.
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns
df[numerical_cols] = df[numerical_cols].fillna(df[numerical_cols].mean())

categorical_cols = df.select_dtypes(include=['object']).columns
df[categorical_cols] = df[categorical_cols].fillna('unknown')
df.head()

In [ ]:
# Task 3: Write your code here:
df.duplicated().sum()
df = df.drop_duplicates()
# df.info()

In [ ]:
# Task 4: Write your code here:
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
# df.info() For info i use them just to visulize i make them comment so i can easily visualize again
# print(categorical_cols)
df

In [ ]:
target = 'Delivery_Time'
feature_cols = df.columns
x = df.drop(target, axis=1)
y = df[target]

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_scale = scaler.fit_transform(x)
# x_scale


In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    x_scale, y,
    test_size=0.2,
    random_state=42
)

print(f"Train: {X_train.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")

In [ ]:
# Task 2,3,4,5: Write your code here: i understanded why its needed in one cell after doing whats below
#KFOLD
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
#Forest
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Model trained")
print('YIPIIIIEEEEEEEEEEE')
print('-'*30)
#init scores
mse_scores = []
mae_scores = []

for train_idx, test_idx in kf.split(x_scale):
    X_train, X_test = x_scale[train_idx], x_scale[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Train model
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Evaluation metrics
    mse_scores.append(mean_squared_error(y_test, y_pred))
    mae_scores.append(mean_absolute_error(y_test, y_pred))

print("=" * 40)

print('model pass?')
print('yes, cool.')
print("=" * 40)

# Print Evaluation Metrics
print("\nModel Evaluation Metrics (K-Fold)\n" + "-"*40)
print(f"MSE : {np.mean(mse_scores):.2f}")
print(f"MAE : {np.mean(mae_scores):.2f}")
print(f"RMSE: {np.sqrt(np.mean(mse_scores)):.2f}")
print("-"*40)
print(np.mean(mae_scores)/5)


In [ ]:
# #Forest
# model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
# model.fit(X_train, y_train)
# print("Model trained")
# print('YIPIIIIEEEEEEEEEEE')

In [ ]:
# #MAE
# y_pred = model.predict(X_test)
# mae = mean_absolute_error(y_test, y_pred)
# print(mae)

In [ ]:
# #acuuracy avg scr
# baseline_pred = [y_train.mean()] * len(y_test)

# baseline_mae = mean_absolute_error(y_test, baseline_pred)
# baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))

# print("=" * 40)
# print("baseline")
# print("=" * 40)
# print(f"Baseline MAE:  {baseline_mae:.4f}")
# print(f"Model MAE:     {mae:.4f}")
# print(f"Improvement:   {baseline_mae - mae:.4f}")
# print("=" * 40)

# if mae < baseline_mae:
#     print("Model beats baseline!")
# else:
#     print("Model worse than baseline - try different model/features")

In [ ]:
df

In [ ]:
# Task 1: Write your code here:
feature_col = ['Distance_km'	,'Weather','Traffic_Level','Time_of_Day',	'Vehicle_Type' ,'Preparation_Time_min','Courier_Experience_yrs']
feature_importance = pd.DataFrame({
    'feature': x.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    "r--",
    linewidth=2
)

plt.xlabel("Actual Exam Scores (Ground Truth)")
plt.ylabel("Predicted Exam Scores")
plt.title("Linear Regression: Predictions vs Ground Truth")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here: